<a href="https://colab.research.google.com/github/shaansuper/ANN_Classification_Project_Churn/blob/main/Student_Result_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
records=pd.read_csv('/content/sample_data/student_performance.csv')
# Explanation for double brackets in pandas:
# When you use single square brackets, like `df['column_name']`, pandas selects a single column and returns it as a Series.
# When you use double square brackets, like `df[['column_name']]` or `df[['col1', 'col2']]`, pandas selects one or more columns and always returns a DataFrame.
# Many machine learning libraries, such as scikit-learn's OneHotEncoder, expect a 2D array-like input (like a DataFrame) even when working with a single feature.
# Therefore, `records[['parent_education']]` ensures the input to the encoder is a DataFrame, which is typically required by such libraries.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle


In [ ]:
records.head()

,student_id,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,STU0001,Male,15,25,63.8,Bachelor,Yes,Yes,41,67,Yes
1,STU0002,Female,15,2,54.7,Bachelor,Yes,Yes,83,28,No
2,STU0003,Female,19,10,90.5,High School,Yes,No,73,49,No
3,STU0004,Male,16,26,66.8,High School,No,Yes,75,70,Yes
4,STU0005,Female,15,25,73.0,High School,No,Yes,67,77,Yes


In [ ]:
records=records.drop(['student_id',],axis=1)

In [ ]:
records.head()

,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,Male,15,25,63.8,Bachelor,Yes,Yes,41,67,Yes
1,Female,15,2,54.7,Bachelor,Yes,Yes,83,28,No
2,Female,19,10,90.5,High School,Yes,No,73,49,No
3,Male,16,26,66.8,High School,No,Yes,75,70,Yes
4,Female,15,25,73.0,High School,No,Yes,67,77,Yes


In [ ]:
label_encoder_gender=LabelEncoder()
records['gender']=label_encoder_gender.fit_transform(records['gender'])
records.head()

,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,1,15,25,63.8,Bachelor,Yes,Yes,41,67,Yes
1,0,15,2,54.7,Bachelor,Yes,Yes,83,28,No
2,0,19,10,90.5,High School,Yes,No,73,49,No
3,1,16,26,66.8,High School,No,Yes,75,70,Yes
4,0,15,25,73.0,High School,No,Yes,67,77,Yes


In [ ]:
label_encoder_internet_access=LabelEncoder()
records['internet_access']=label_encoder_internet_access.fit_transform(records['internet_access'])


In [ ]:
label_encoder_extracurricular=LabelEncoder()
records['extracurricular']=label_encoder_extracurricular.fit_transform(records['extracurricular'])

In [ ]:
label_encoder_passed=LabelEncoder()
records['passed']=label_encoder_passed.fit_transform(records['passed'])
records.head()


,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,1,15,25,63.8,Bachelor,1,1,41,67,1
1,0,15,2,54.7,Bachelor,1,1,83,28,0
2,0,19,10,90.5,High School,1,0,73,49,0
3,1,16,26,66.8,High School,0,1,75,70,1
4,0,15,25,73.0,High School,0,1,67,77,1


In [ ]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder=OneHotEncoder()

In [ ]:
education_encoder=one_hot_encoder.fit_transform(records[['parent_education']])

In [ ]:
education_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 500 stored elements and shape (500, 5)>

In [ ]:
one_hot_encoder.get_feature_names_out(['parent_education'])

array(['parent_education_Bachelor', 'parent_education_High School',
       'parent_education_Master', 'parent_education_PhD',
       'parent_education_nan'], dtype=object)

In [ ]:
encoded_df=pd.DataFrame(education_encoder.toarray(),columns=one_hot_encoder.get_feature_names_out(['parent_education']))

In [ ]:
encoded_df.head()

,parent_education_Bachelor,parent_education_High School,parent_education_Master,parent_education_PhD,parent_education_nan
0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0


In [ ]:
records.head()

,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,1,15,25,63.8,Bachelor,1,1,41,67,1
1,0,15,2,54.7,Bachelor,1,1,83,28,0
2,0,19,10,90.5,High School,1,0,73,49,0
3,1,16,26,66.8,High School,0,1,75,70,1
4,0,15,25,73.0,High School,0,1,67,77,1


In [ ]:
records.drop(['parent_education'],axis=1,inplace=True)

In [ ]:
records=pd.concat([records,encoded_df],axis=1)

In [ ]:
records.head()

,gender,age,study_hours_per_week,attendance_rate,internet_access,extracurricular,previous_score,final_score,passed,parent_education_Bachelor,parent_education_High School,parent_education_Master,parent_education_PhD,parent_education_nan
0,1,15,25,63.8,1,1,41,67,1,1.0,0.0,0.0,0.0,0.0
1,0,15,2,54.7,1,1,83,28,0,1.0,0.0,0.0,0.0,0.0
2,0,19,10,90.5,1,0,73,49,0,0.0,1.0,0.0,0.0,0.0
3,1,16,26,66.8,0,1,75,70,1,0.0,1.0,0.0,0.0,0.0
4,0,15,25,73.0,0,1,67,77,1,0.0,1.0,0.0,0.0,0.0


In [ ]:
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder_gender,file)

with open('label_encoder_passed.pkl','wb') as file:
  pickle.dump(label_encoder_passed,file)

with open('one_hot_encoder.pkl','wb') as file:
  pickle.dump(one_hot_encoder,file)

with open('label_encoder_extracurricular.pkl','wb') as file:
  pickle.dump(label_encoder_extracurricular,file)

with open('label_encoder_internet_access.pkl','wb') as file:
  pickle.dump(label_encoder_internet_access,file)

In [ ]:
#Split data into dependent and independent

X=records.drop('passed',axis=1) #Independent
y=records['passed'] #Dependent


X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

with open('scaler.pkl','wb') as file:
  pickle.dump(scaler,file)

In [ ]:
records.head()

,gender,age,study_hours_per_week,attendance_rate,internet_access,extracurricular,previous_score,final_score,passed,parent_education_Bachelor,parent_education_High School,parent_education_Master,parent_education_PhD,parent_education_nan
0,1,15,25,63.8,1,1,41,67,1,1.0,0.0,0.0,0.0,0.0
1,0,15,2,54.7,1,1,83,28,0,1.0,0.0,0.0,0.0,0.0
2,0,19,10,90.5,1,0,73,49,0,0.0,1.0,0.0,0.0,0.0
3,1,16,26,66.8,0,1,75,70,1,0.0,1.0,0.0,0.0,0.0
4,0,15,25,73.0,0,1,67,77,1,0.0,1.0,0.0,0.0,0.0


### **ANN Implementation**

1. Create Sequence Model
2. Use Dense Library of Keras to create nodes
3. Define Activation Function(e.g- Relu, Sigmoid etc.)

4. Define the Optimizer (e.g:- SGD,Adam) and carry out Forward and Backward Propagation to adjust weights

5. Define/Calculate Loss function (e.g:- Mean Squared Error, Cross Entropy) and minimize it during training

6. Evaluate model's performance on testing/validation using parameters like Accuracy.

In [ ]:
import tensorflow as tf
from  tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import datetime



In [ ]:
Avishek_Prediction_Model=Sequential([
      Dense(16,activation='relu',input_shape=(X_train.shape[1],)),
      Dense(8,activation='relu'),
      Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Set Optimizers,Loss Function

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
opt=Adam(learning_rate=0.01)
loss_function=BinaryCrossentropy()
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
Avishek_Prediction_Model.compile(loss=loss_function,optimizer=opt,metrics=['accuracy'])


In [ ]:
#Setup tensor Board

logs_directory="logs/fit"+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

In [ ]:
tensor_flow_callback=TensorBoard(log_dir=logs_directory,histogram_freq=1)


In [ ]:
history=Avishek_Prediction_Model.fit(X_train,y_train,validation_data=(X_test,y_test),
                                     epochs=100,callbacks=[tensor_flow_callback,early_stopping_callback])

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7425 - loss: 0.4894 - val_accuracy: 0.9100 - val_loss: 0.3413
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8725 - loss: 0.3024 - val_accuracy: 0.9400 - val_loss: 0.1860
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9250 - loss: 0.1920 - val_accuracy: 0.9600 - val_loss: 0.1158
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9450 - loss: 0.1311 - val_accuracy: 0.9500 - val_loss: 0.1020
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9700 - loss: 0.0944 - val_accuracy: 0.9600 - val_loss: 0.0839
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9875 - loss: 0.0644 - val_accuracy: 0.9600 - val_loss: 0.0716
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9900 - loss: 0.0477 - val_accuracy: 0.9600 - val_loss: 0.0589
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9875 - loss: 0.0372 - val_accuracy: 0.980

In [ ]:
Avishek_Prediction_Model.save('Avishek_Prediction_Model.keras')